---

# Data Curation

---

## Introduction

The aim of this notebook is to collect and organise medical literature relating to autoimmune gastritis (AIG) and pernicious (PA) for further NLP analysis. Articles are retrieved from PubMedCentral (PMC) Open Access using the NCBI E-utilities API, with searches restricted to titles and abstracts to improve relevance. Full-text articles are then downloaded using the BioC API and stored in JSON format to preserve their text and structure. 

The NCBI E-utilities API was selected because it allows biomedical literature to be searched and retrieved automatically using specific search queries. In particular, the ESearch utility returns article identifiers that match the search criteria, allowing relevant PMC articles to be identified without manually searching for each one (Sayers, 2022).

The BioC API was used to retrieve the full-text articles in JSON format. BioC provides a structured representation of biomedical articles that is designed to support automated text processing and information extraction (Comeau et al., 2019). This is useful for the project because the article text, metadata and section information can be preserved and organised for further NLP analysis.

## Obtaining the Data

First, we can load in the relevant libraries.

In [2]:
# import relevant libraries 
import json
import time
import requests
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

In [ ]:
# We will download 10 articles per condition 
# We can increase this later
N = 10 

# We can set up the folders for the downloaded JSON files and
# processed files
JSON_DIR = Path("pmc/json")
OUTPUT_DIR = Path("pmc/processed")

# We can create the folders if they do not exist yet
JSON_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# NCBI E-utilities API URL used to search PMC and retrieve article IDs
SEARCH_URL = (
    "https://eutils.ncbi.nlm.nih.gov/"
    "entrez/eutils/esearch.fcgi"
)

# BioC API URL used to download the full text of each article as JSON
# The {pmcid} will be replaced with the article's PMC ID
BIOC_URL = (
    "https://www.ncbi.nlm.nih.gov/research/bionlp/"
    "RESTful/pmcoa.cgi/BioC_json/{pmcid}/unicode"
)

# We can define the search terms for autoimmune gastritis (AIG)
# and pernicious anaemia (PA), including alternative disease names

# Search for Open Access articles discussing diagnosis
#search_queries = {
#   "DIAGNOSTIC": (
#        '(diagnosis[tiab] OR diagnostic[tiab]) '
#        'AND "open access"[filter]'
#   )
#}

search_queries = {
    "AIG": (
        '("autoimmune gastritis"[tiab] OR '
        '"autoimmune atrophic gastritis"[tiab] OR '
        '"autoimmune metaplastic atrophic gastritis"[tiab] OR '
        '"autoimmune chronic atrophic gastritis"[tiab]) '
        'AND "open access"[filter]'
    ),

    "PA": (
        '("pernicious anemia"[tiab] OR '
        '"pernicious anaemia"[tiab] OR '
        '"Biermer disease"[tiab]) '
        'AND "open access"[filter]'
    )
}

# [tiab] limits the search to article titles and abstracts
# The open access filter limits the results to PMC Open Access articles

# We can create a session to reuse the connection when making API requests
session = requests.Session()

In [4]:
# We can define a function to search the PMC for articles
# matching the search query 
# max_results sets the maximum number of article IDs returned
def search_pmc(query, max_results=100):

    # We limit the search to aeticles published between
    # 2015 and 2025 for now 
    # Recent papers (2026) do not load in correctly
    query = (
        f"({query}) AND "
        '("2015/01/01"[PDAT] : "2025/12/31"[PDAT])'
    )

    # Send a request to the NCBI E-utilities API
    response = session.get(
        SEARCH_URL,
        params={
            "db": "pmc", # search the PMC database
            "term": query, # use the search terms defined earlier
            "retmax": max_results, # maximum number of article IDs to return
            "retmode": "json", # return the results in JSON format
            "sort": "relevance" # sort articles by relevance
        },
        timeout=30 # stop the request if it takes over 30 seconds
    )

    # check whether the API request was successful
    # raises an error if the server returns a failed HTTP status
    response.raise_for_status()

    # convert the JSON response into a python dictionary 
    # extract the search results from the response
    result = response.json()["esearchresult"]

    # print the total number of articles matching the search query
    print("PMC search matches:", result["count"])

    # return the article IDs with PMC added to each one
    # we will use these IDs to download the full-text articles
    return ["PMC" + x for x in result["idlist"]]

In [5]:
# We can create a function to download the full-text articles from the
# BioC API. 
# Uses the PMC ID to retrieve the article in JSON format. 
def download_bioc(pmcid):
    
    # Replace {pmcid} in the BioC url with the article's PMC ID
    url = BIOC_URL.format(pmcid=pmcid)
    
    # send a request to the BioC API
    # Stop the request if it takes longer than 60 seconds
    response = session.get(url, timeout=60)
    
    # check whether the request was successful
    response.raise_for_status()

    # try to convert the response into JSON
    try:
        data = response.json()
    
    # if the response is not valid, return an error
    # we can show first 150 chars to help identify issue
    except ValueError:
        raise ValueError(
            f"Invalid JSON response: {response.text[:150]!r}"
        )

    # store data as a list 
    collections = data if isinstance(data, list) else [data]

    # check whether the downloaded articles contains text passages
    # as some articles may not have full text available through BioC
    # (for example, recent papers from 2026)
    has_passages = any(
        document.get("passages")
        for collection in collections
        if isinstance(collection, dict)
        for document in collection.get("documents", [])
    )

    # Skip the articule if no passages were found
    if not has_passages:
        raise ValueError("No full-text passages returned")

    # return the JSON data 
    return data

In [6]:
# Create a list to keep track of downloaded articles and their details
records = []

# Use a set to keep track of unique PMC IDs and avoid counting duplicates
downloaded = set()

# Go through the search queries for AIG and PA separately
for condition, query in search_queries.items():

    # Show which condition is currently being searched
    print(f"\nSearching for {condition}...")

    # Search PMC to retrieve a list of possible articles
    # Retrieve extra IDs in case some articles are unavailable through BioC
    candidate_ids = search_pmc(
        query,
        max_results=max(50, N * 3)
    )

    # Keep track of how many articles have been successfully processed
    successes = 0

    # Go through the article IDs and display a progress bar
    for pmcid in tqdm(candidate_ids, desc=condition):

        # Stop once the target number of articles has been reached
        if successes >= N:
            break

        # Set the file path using the article's PMC ID
        file_path = JSON_DIR / f"{pmcid}.json"

        try:
            # Check whether the article has already been downloaded
            # This avoids downloading the same article again
            if not file_path.exists():

                # Download the full-text article from the BioC API
                data = download_bioc(pmcid)

                # Save the article as a JSON file
                # Keep the original text and structure for later NLP analysis
                file_path.write_text(
                    json.dumps(data, ensure_ascii=False, indent=2),
                    encoding="utf-8"
                )

            # Add the PMC ID to the set if it hasn't been counted yet
            if pmcid not in downloaded:
                downloaded.add(pmcid)

            # Record the article's PMC ID, condition and saved file location
            # This information will be used to create the article manifest
            records.append({
                "pmcid": pmcid,
                "condition": condition,
                "file": str(file_path)
            })

            # Increase the count after successfully processing an article
            successes += 1

        # Handle errors if an article cannot be downloaded or read
        # Print the reason and continue with the next article
        except (requests.RequestException,
                ValueError,
                json.JSONDecodeError) as e:
            print(f"Skipped {pmcid}: {e}")

        # Add a short delay between requests to avoid overloading NCBI servers
        time.sleep(0.4)

    # Print how many articles were successfully processed for each condition
    print(f"{condition}: {successes} articles saved")



Searching for AIG...
PMC search matches: 285


AIG:   0%|          | 0/50 [00:00<?, ?it/s]

Skipped PMC7699377: 429 Client Error: Too Many Requests for url: https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/PMC7699377/unicode
AIG: 10 articles saved

Searching for PA...
PMC search matches: 278


PA:   0%|          | 0/50 [00:00<?, ?it/s]

Skipped PMC5581415: 429 Client Error: Too Many Requests for url: https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/PMC5581415/unicode
PA: 10 articles saved


In [7]:
# Convert the list of downloaded article records into a DataFrame
# This contains the PMC ID, condition and file location
manifest = pd.DataFrame(records)

# Save the DataFrame as a CSV file
# This allows us to keep track of which articles were downloaded
manifest.to_csv(
    OUTPUT_DIR / "article_manifest.csv",
    index=False  # Don't include the DataFrame index in the CSV
)

# Print a message to confirm the download process has finished
print("\nDownload complete!")

# Show the number of unique articles downloaded
# Articles matching both AIG and PA are only counted once
print("Unique articles:", len(downloaded))

# Show the full folder path where the JSON articles are stored
print("JSON directory:", JSON_DIR.resolve())


Download complete!
Unique articles: 20
JSON directory: C:\Users\caral\OneDrive\Documents\SCI501\SCI501\pmc\json


## Check Downloaded Data

In [8]:
# Get a list of all JSON files saved in the download folder
json_files = list(JSON_DIR.glob("*.json"))

# Print the total number of JSON articles
print("Downloaded JSON files:", len(json_files))

# Show an example file name to check that articles were saved (and saved correctly)
# If no files found, print "None"
print("Example filename:", json_files[0].name if json_files else "None")

Downloaded JSON files: 20
Example filename: PMC10474957.json


In [9]:
# Open the first downloaded JSON article
# Use UTF-8 encoding to read the text correctly as the articles
# may have special characters
with open(json_files[0], encoding="utf-8") as f:

    # Load the JSON file into a Python object
    #  so we can inspect the article's text, metadata and structure
    article = json.load(f)

# Display the article to see how the BioC data is organised
article

[{'source': 'PMC',
  'date': '20230923',
  'key': 'pmc.key',
  'infons': {},
  'documents': [{'id': '10474957',
    'infons': {'license': 'CC BY'},
    'passages': [{'offset': 0,
      'infons': {'article-id_doi': '10.1155/2023/2747756',
       'article-id_pmc': '10474957',
       'article-id_pmid': '37663274',
       'elocation-id': '2747756',
       'license': 'This is an open access article distributed under the Creative Commons Attribution License, which permits unrestricted use, distribution, and reproduction in any medium, provided the original work is properly cited.',
       'name_0': 'surname:Kamada;given-names:Kentaro',
       'name_1': 'surname:Kawano;given-names:Osamu',
       'name_2': 'surname:Yakuwa;given-names:Satoshi',
       'name_3': 'surname:Wakasa;given-names:Kentaro',
       'name_4': 'surname:Uetake;given-names:Kimiaki',
       'section_type': 'TITLE',
       'type': 'front',
       'volume': '2023',
       'year': '2023'},
      'text': 'Pernicious Anemia in an 

## Sources

---

Sayers, E. W. (2022). A general introduction to the E-utilities. In Entrez programming utilities help. National Center for Biotechnology Information. https://www.ncbi.nlm.nih.gov/books/NBK25497/

Comeau, D. C., Wei, C.-H., Islamaj Doğan, R., & Lu, Z. (2019). PMC text mining subset in BioC: About three million full-text articles and growing. Bioinformatics, 35(18), 3533–3535. https://doi.org/10.1093/bioinformatics/btz070